# 从「小猫坐在」到下一个字：一步步看懂 GPT

输入「小猫坐在」，模型怎样给「垫」「门」等字符分配概率？这个问题会带我们走过整个 Transformer，而后我们再用文字教它学会预测。

这里的 **GPT** 指采用因果自注意力的、只含 Decoder 的语言模型。我们会自己写一个很小的版本，不调用聊天接口，也不下载预训练模型。它只在几句练习文本上训练；我们观察它怎样记住这些模式，不据此判断它是否能理解或泛化。

**阅读方式**：在现有 Docker 的 Jupyter 中打开并 Trust，然后从干净内核按顺序运行。每节先读问题、预测实验结果，再运行代码核对。所有代码和交互都在本文件中；折叠的只是重复绘图工具，关键计算会展开。CPU 即可，主线不需要联网。控件操作需要保持本 Notebook 的内核运行。

**贯穿例子**：`小猫坐在`。为了看清每一步，我们把**一个字符当作一个 token**；因此这里的“下一个 token”就是“下一个字符”。真实模型常用子词 token，机制放在最后对照。

| 章节 | 我们要解释的问题 |
|---|---|
| 1. 全景 | 模型输入什么，输出什么，怎样循环续写？ |
| 2. 输入 | 字符、编号、向量和位置各有什么用？ |
| 3. Attention | 一个位置怎样把别的位置的信息取回来？ |
| 4. Block | 汇集信息之后，还需要哪些加工？ |
| 5. 输出 | 隐藏向量怎样变成词表上的概率？ |
| 6. 学习 | 答案从哪来，错误怎样改变参数？ |
| 7. 训练 | 同一套模型学前和学后有什么变化？ |
| 8. 对照 | GPT、BERT 和翻译用的 Transformer 有什么关系？ |


In [ ]:
import copy
import html
import math
from collections import OrderedDict

import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
import ipywidgets as widgets
from IPython.display import HTML, display, clear_output

# 这只是本内核的 CPU 设置；小模型通常不需要大量线程。
torch.set_num_threads(min(2, torch.get_num_threads()))
torch.manual_seed(7)
np.set_printoptions(precision=3, suppress=True)
print(f"环境就绪：PyTorch {torch.__version__}；CPU；无外部模型下载")


In [ ]:
# 可折叠工具：只负责显示。后面的数学与模型计算均在正文展开。
PANELS = {}
FLOW = ['字符与 ID', '输入向量 + 位置', 'Transformer Blocks', '词表分数 / 概率', '选择并接回']

def array(value):
    return value.detach().cpu().numpy() if isinstance(value, torch.Tensor) else np.asarray(value)

def number(value):
    if isinstance(value, (float, np.floating)):
        return f'{value:.3f}'
    return str(value)

def table(headers, rows, title=None):
    th = ''.join(f'<th style="text-align:left;padding:6px;border-bottom:2px solid #ccd;white-space:nowrap">{html.escape(str(x))}</th>' for x in headers)
    body = ''.join('<tr>'+''.join(f'<td style="padding:6px;border-bottom:1px solid #eef;white-space:nowrap">{html.escape(number(x))}</td>' for x in row)+'</tr>' for row in rows)
    title_html = f'<b>{html.escape(title)}</b>' if title else ''
    display(HTML(f'<div style="overflow:auto;margin:10px 0">{title_html}<table style="font-variant-numeric:tabular-nums;text-align:left">{th}{body}</table></div>'))

def vector_grid(values, labels, title, limit=8):
    a = array(values)
    if a.ndim == 1:
        a = a[None, :]
    visible = a[:, :limit]
    peak = max(float(np.abs(visible).max()), 1e-8)
    rows = []
    for label, row in zip(labels, visible):
        tds = ''
        for x in row:
            alpha = min(.8, abs(float(x))/peak*.65)
            rgb = '47,114,190' if x >= 0 else '219,113,47'
            tds += f'<td style="padding:7px;white-space:nowrap;background:rgba({rgb},{alpha:.3f});color:#111">{x:.3f}</td>'
        rows.append(f'<tr><th style="padding:7px;text-align:left;white-space:nowrap">{html.escape(str(label))}</th>{tds}</tr>')
    display(HTML(f'<div style="overflow:auto;margin:8px 0"><b>{html.escape(title)}</b>（形状 {a.shape}；显示前 {visible.shape[1]} 列）<table style="font-variant-numeric:tabular-nums">'+''.join(rows)+'</table><small>蓝为非负，橙为负；颜色只比较当前表内的数值大小。</small></div>'))

def bars(values, labels, title, top=None):
    a = array(values).reshape(-1)
    order = np.argsort(-a)[:top] if top else np.arange(len(a))
    rows = []
    for i in order:
        p = float(a[i])
        rows.append(f'<div style="display:flex;align-items:center;gap:10px;margin:4px 0;min-width:425px"><span style="width:85px">{html.escape(str(labels[i]))}</span><div style="width:260px;background:#f0f3f7"><div style="width:{max(0,min(100,p*100)):.2f}%;height:16px;background:#3c78b4"></div></div><span>{p:.3f}</span></div>')
    display(HTML(f'<div style="padding:10px;border:1px solid #dde;margin:10px 0;overflow:auto"><b>{html.escape(title)}</b>'+''.join(rows)+'</div>'))

def roadmap(active):
    boxes = []
    for i, label in enumerate(FLOW):
        color = '#dceeff' if i in active else '#f4f5f7'
        boxes.append(f'<span style="display:inline-block;padding:8px;background:{color};border:1px solid #ccd;border-radius:5px;margin:3px">{html.escape(label)}</span>')
    display(HTML('<div style="line-height:2">'+' → '.join(boxes)+'</div>'))

def loss_plot(losses):
    vals = np.asarray(losses, dtype=float)
    lo, hi = float(vals.min()), float(vals.max())
    span = max(hi-lo, .01)
    points = ' '.join(f'{40+i/max(len(vals)-1,1)*590:.2f},{190-(float(y)-lo)/span*150:.2f}' for i,y in enumerate(vals))
    return f'<svg role="img" aria-label="训练批次平均损失曲线" viewBox="0 0 680 235" style="max-width:700px;width:100%;background:#fafcff"><path d="M40 25 V195 H640" fill="none" stroke="#aaa"/><polyline points="{points}" fill="none" stroke="#3c78b4" stroke-width="2"/><text x="45" y="20" font-size="13">批次 loss（不是泛化评估）</text><text x="3" y="45" font-size="11">{hi:.2f}</text><text x="3" y="193" font-size="11">{lo:.2f}</text><text x="40" y="220" font-size="11">1</text><text x="570" y="220" font-size="11">step {len(vals)}</text></svg>'

def panel(key, render, controls):
    out = widgets.Output()
    state = {'controls': controls, 'renders': 0, 'error': None}
    def refresh(change=None):
        state['renders'] += 1
        state['error'] = None
        with out:
            clear_output(wait=True)
            try:
                render(**{k: v.value for k,v in controls.items()})
            except ValueError as exc:
                state['error'] = str(exc)
                display(HTML(f'<p style="color:#b34">{html.escape(str(exc))}</p>'))
    for control in controls.values():
        control.observe(refresh, names='value')
    state['refresh'] = refresh
    PANELS[key] = state
    refresh()
    ui = widgets.VBox([widgets.VBox(list(controls.values())), out])
    display(ui)
    return ui


## 1 · 先看全景：一次只续出一个 token，为什么能写一整句？

**我们现在在哪**：先把整个生成循环摆在桌上，暂时把中间的 Block 当作一个待拆开的盒子。

给定「小猫坐在」，语言模型并不直接吐出一句“正确答案”。它先为**词表里的每个字符**算一个分数，再把分数变成概率，最后选择一个字符。这个字符接回输入，整个过程再运行一次：

`小猫坐在 → 下一字概率 → 选出「垫」→ 小猫坐在垫 → 再算概率 → ……`

这里的「垫」只是说明循环的例子，尚未训练的模型不一定选它。生成时参数固定，变化的是输入和中间计算结果；训练时我们才改参数。

先认识两类容易混淆的量：**参数**是模型学到并反复使用的规则，比如查表的内容和线性变换的矩阵；**中间表示**是这次输入产生的向量，换输入就可能改变。模型会为每个位置产生输出，续写只取**最后一个位置**的词表分数。第 6 节再解释其他位置的输出在训练中怎样派上用场。


In [ ]:
roadmap({0,1,2,3,4})
# 手工分数，仅演示“分数 → 概率”，不是任何模型的预测。
illustration_chars = ['垫', '门', '天', '池', '。']
illustration_logits = torch.tensor([2., 1., 0., -1., -.5])
bars(torch.softmax(illustration_logits, dim=-1), illustration_chars,
     '手工示意：只在这 5 个候选中归一化；真实模型覆盖完整词表')


**先预测**：续写三个字符，是只运行一次模型，还是运行三次？每次需要重新训练吗？

<details><summary>核对与下一步</summary>本教程不用 KV cache，每生成一个字符都重新运行一次前向计算，因此是三次。模型参数不变。下一步先拆开入口：文字怎样进入这个只会算数的盒子？</details>


## 2 · 文字怎样进入模型：编号是地址，向量才是可加工的表示

**位置**：字符 → token ID → 输入向量。**问题**：矩阵运算不能直接处理文字，也不能把任意编号的数值距离当成文字关系。

先从我们的练习文本收集一个词表。一个字符一个编号，只是在词表中给它一个地址：「猫」的 ID 比「狗」小，不代表猫比较小或比较相似。随后用 ID 查 Embedding 表，取出一行可学习的数值。训练会改变这张表，编号本身不参与梯度更新。

向量可以先理解为“一排数”；矩阵是“把多排数叠起来”。我们把序列排成矩阵，每行一个位置，每列一个特征坐标。坐标的意义由学习共同形成，不能事先把某一列指定为“猫的程度”。


In [ ]:
roadmap({0,1})
SENTENCES = [
    '小猫坐在垫子上。', '小狗坐在门口。',
    '小猫躺在垫子上。', '小狗趴在门口。',
    '小鸟飞在天空中。', '小鱼游在池水里。',
]
CORPUS = ''.join(SENTENCES) * 40
CHARS = list(dict.fromkeys(CORPUS))
STOI = {ch: i for i, ch in enumerate(CHARS)}
ITOS = dict(enumerate(CHARS))
VOCAB_SIZE = len(CHARS)
PREFIX = '小猫坐在'

def encode(text):
    if not text:
        raise ValueError('请输入至少一个字符。')
    unknown = sorted(set(text) - set(STOI))
    if unknown:
        raise ValueError('字符不在本练习词表中：'+ '、'.join(repr(ch) for ch in unknown))
    return torch.tensor([[STOI[ch] for ch in text]], dtype=torch.long)

def decode(ids):
    return ''.join(ITOS[int(i)] for i in ids)

prefix_ids = encode(PREFIX)
table(['位置 t（从 0 开始）', '字符 / token', '词表 ID'],
      [(t, ch, STOI[ch]) for t,ch in enumerate(PREFIX)], '同一个贯穿前缀')
print('练习词表：', ''.join(CHARS), '；大小 V =', VOCAB_SIZE)
assert decode(prefix_ids[0]) == PREFIX


### 2.1 同一个字在不同位置，为什么还要加一个向量？

单独查词表时，同一个 ID 总会取到同一行。输入中的「在」还没读过前面的「小猫」，也没有自己的位置标记。为此我们再查一张**位置表**，把位置向量与字符向量相加：

$$x_t = E[\mathrm{id}_t] + P[t]$$

`E` 按词表 ID 查，`P` 按位置查；相加后仍是一排同样长的数。这个模型的两张表都由训练学习。位置标记告诉后面的计算“这行来自第几位”，上下文信息则要等 Attention 来汇集。

下面使用四维**手工教具**。它只方便我们看数，不是学好的语义向量，也不是后面 64 维模型的参数。


In [ ]:
toy_rng = np.random.default_rng(12)
toy_E = toy_rng.normal(0, .3, (VOCAB_SIZE, 4))
# 贯穿前缀的四个字符使用便于计算的数值。
toy_E[[STOI[ch] for ch in PREFIX]] = np.array([
    [1.,0.,0.,0.], [0.,1.,0.,0.], [0.,0.,1.,0.], [1.,1.,0.,0.]])
toy_P = np.array([[0.,0.,0.,0.], [.1,0.,0.,0.], [0.,.1,0.,0.], [0.,0.,.1,0.]])
toy_tokens = toy_E[prefix_ids[0].numpy()]
toy_X = toy_tokens + toy_P
vector_grid(toy_tokens, list(PREFIX), '手工字符向量 E[id]')
vector_grid(toy_P, [f'位置 {i}' for i in range(4)], '手工位置向量 P[t]')
vector_grid(toy_X, list(PREFIX), '相加后，作为 Block 的输入 X')
print('X 的形状：', toy_X.shape, '=（位置数 T，向量维度 D）')


In [ ]:
def inspect_toy_input(position):
    ch = PREFIX[position]
    table(['步骤', '对应内容'], [
        ['当前位置 / ID', f'{position} / {ch!r} / {STOI[ch]}'],
        ['字符向量', toy_tokens[position].round(3).tolist()],
        ['位置向量', toy_P[position].round(3).tolist()],
        ['相加后的输入', toy_X[position].round(3).tolist()],
    ])

input_ui = panel('input', inspect_toy_input, OrderedDict(
    position=widgets.Dropdown(options=[(f'{i}：{ch}',i) for i,ch in enumerate(PREFIX)], value=3, description='查看位置')))


**先预测**：如果把「猫」挪到另一位置，它查到的字符向量会变吗？相加后的输入向量呢？

<details><summary>核对与下一步</summary>字符 ID 没变，字符向量不变；位置变了，位置向量通常不同，因此相加后的输入会变。这仍没有读到其他字符。接下来要解决的是：怎样让当前位置的信息和前文发生关系？</details>

<details><summary>选读：One-hot 和位置的数学说法</summary>
One-hot 是只有对应 ID 那一项为 1 的长向量。One-hot 乘 Embedding 表，等价于直接取出那一行；实际实现用查表避免构造大量的 0。没有位置输入、没有因果 Mask 的全自注意力具有置换等变性：重排输入各行，输出也随之重排，并非输出完全不变。GPT 的因果 Mask 也提供了访问结构限制；我们仍显式学习位置向量，帮助模型利用位置。
</details>


## 3 · Attention：把“别的位置有什么”写入当前位置

**位置**：进入第一个 Block，先处理位置之间的信息流。**问题**：「小猫坐在」和「小狗坐在」里的「在」，输入字符相同、位置相同，如果一直逐位置独立计算，就无法根据前面的动物调整下一字预测。

我们需要一次跨位置的信息汇集：为当前位置分配一组权重，决定从哪些位置各取多少内容，再把这些内容相加。**Attention 的输出仍然是向量，不是一个被选中的词，也不是答案文本。**

先暂时指定一组权重，理解“取回来”这一步。我们把每个位置提供的内容记作 `V`（Value）。本教具从四维输入投影出两维内容；每个位置用同一个矩阵变换，但它们的输入不同。


In [ ]:
roadmap({2})
toy_WV = np.array([[1.,0.], [0.,1.], [1.,1.], [-1.,1.]])
toy_V = toy_X @ toy_WV
hand_weights = np.array([.1, .6, .1, .2])   # 暂时人为分配给最后位置「在」
hand_mixed = hand_weights @ toy_V
vector_grid(toy_V, list(PREFIX), '同一个输入 X 变换成每个位置提供的内容 V')
table(['来源', '给它的权重', 'V 的第 0 维', 'V 的第 1 维', '贡献第 0 维', '贡献第 1 维'],
      [(ch, float(w), *v.tolist(), *(w*v).tolist()) for ch,w,v in zip(PREFIX,hand_weights,toy_V)])
print('汇集结果 = 0.1 V[小] + 0.6 V[猫] + 0.1 V[坐] + 0.2 V[在] =', hand_mixed)


### 3.1 权重从哪里来？把“匹配”和“内容”分成不同投影

刚才权重由我们填写。模型需要根据输入自己计算权重，因此从同一个输入矩阵 `X` 产生三组向量：

$$Q=XW_Q,\qquad K=XW_K,\qquad V=XW_V$$

- **Q（Query）**：当前位置拿什么特征去匹配其他位置。
- **K（Key）**：每个位置拿什么特征供别人匹配。
- **V（Value）**：匹配之后，每个位置实际贡献什么内容。

三个矩阵是可以学习的参数，并非人为给各个字填写“问题、标签、答案”。同一个字在后续层中已经包含上下文，这三组向量也会相应改变。

对最后位置「在」，先拿它的 Q 与每个位置的 K 做点积：两个向量对应坐标相乘再求和。**分数用于决定取多少，V 决定取回什么**。分开投影，让匹配用的特征不必和传递内容的特征完全相同。投影可学习，点积也不是天然的语义相似度判官。


In [ ]:
toy_WQ = np.array([[1.,0.], [0.,1.], [1.,0.], [0.,1.]])
toy_WK = np.array([[1.,0.], [0.,1.], [0.,1.], [1.,0.]])
toy_Q = toy_X @ toy_WQ
toy_K = toy_X @ toy_WK
# 所有位置同时匹配：第 i 行、第 j 列 = Q[i] 与 K[j] 的点积。
toy_raw_scores = toy_Q @ toy_K.T
toy_scores = toy_raw_scores / math.sqrt(toy_Q.shape[-1])
vector_grid(toy_Q, list(PREFIX), 'Q：每个位置用于查询的向量')
vector_grid(toy_K, list(PREFIX), 'K：每个位置供匹配的向量')
table(['被查询的位置', 'Q[在] · K[j]', '除以 √2 后'],
      [(ch, toy_raw_scores[3,j], toy_scores[3,j]) for j,ch in enumerate(PREFIX)],
      f'Q[在] = {toy_Q[3].tolist()}；分数仍可为负，也不要求和为 1')


### 3.2 把匹配分数变成权重：Softmax 与因果 Mask

我们希望对 V 做加权平均，因此需要非负、总和为 1 的权重。标准 Transformer 用 Softmax：先对每个分数取指数，再除以指数总和。

$$a_{tj}=\frac{\exp(s_{tj})}{\sum_{j'}\exp(s_{tj'})},\qquad z_t=\sum_j a_{tj}v_j$$

例如分数 `(2, 1, 0)` 变成约 `(0.665, 0.245, 0.090)`。高分获得更多权重，其他位置仍可贡献内容；这种平滑计算也便于训练。这里 Softmax 的选项是**输入位置**，不是词表里的字符。

我们还必须限制：**位置 t 的输出预测 t+1，只能读取位置 ≤ t 的输入**。训练时整句文本已经在手中，如果「坐」能读到后面的「在」，预测下一字就变成抄答案了。因此，在归一化之前把未来位置的分数设为负无穷；指数变成 0，这些位置便不参与加权。

完整的一头注意力是：

$$Z=\operatorname{softmax}_{\text{每行}}\left(\frac{QK^\top}{\sqrt{d_k}}+M\right)V$$

`M` 在允许读取的位置为 0，未来位置为负无穷。除以 `√d_k` 是为了控制点积分数随维度增加而扩大的典型尺度，减少 Softmax 过早变得极尖的情况；它不是概率归一化。


In [ ]:
def softmax_np(scores):
    # 每行减最大值：指数的公共倍数会在归一化中消掉，结果不变且更稳定。
    shifted = scores - scores.max(axis=-1, keepdims=True)
    exps = np.exp(shifted)
    return exps / exps.sum(axis=-1, keepdims=True)

def toy_attention(query=3, key=1, delta=0., causal=True):
    scores = toy_scores.copy()
    scores[query, key] += delta  # 控制实验：只改变指定的一条匹配分数。
    if causal:
        scores = np.where(np.tril(np.ones_like(scores, dtype=bool)), scores, -np.inf)
    weights = softmax_np(scores)
    mixed = weights @ toy_V
    return scores, weights, mixed

scores, weights, mixed = toy_attention()
print('softmax(2,1,0) =', softmax_np(np.array([[2.,1.,0.]]))[0])
vector_grid(weights, list(PREFIX), '行是 query 位置，列依次是 小 / 猫 / 坐 / 在；每行和为 1')
vector_grid(mixed, list(PREFIX), '每行是该位置取回的两维内容，仍然不是字符')
assert np.allclose(weights.sum(axis=-1), 1.)
assert np.all(weights[np.triu_indices(4, 1)] == 0.)


### 3.3 连续实验：修改一条匹配分数，取回的内容怎样变化？

先选 query 位置，再选一条来源位置，提高或降低它的分数。观察同一张表里的分数、权重、V 和实际贡献，最后看加总结果。

**先预测**：选「坐」作为 query，选未来的「在」作为来源。保持因果 Mask 开启，即使把这条分数提高 3，输出会变吗？再关闭 Mask 会怎样？最后一行的 query 没有未来位置，开关 Mask 不会改变它。


In [ ]:
def render_attention(query, key, delta, causal):
    scores, weights, mixed = toy_attention(query, key, delta, causal)
    table(['来源位置', '归一化前分数', '权重', 'V0', 'V1', '权重 × V0', '权重 × V1'], [
        [f'{j}：{ch}', '被 Mask' if not np.isfinite(scores[query,j]) else scores[query,j],
         weights[query,j], *toy_V[j].tolist(), *(weights[query,j]*toy_V[j]).tolist()]
        for j,ch in enumerate(PREFIX)])
    bars(weights[query], [f'{j}：{ch}' for j,ch in enumerate(PREFIX)], f'位置 {query}「{PREFIX[query]}」给输入位置的权重')
    print('把表中最后两列分别相加，得到：', mixed[query].round(4))
    print('这些数将送进后续投影和残差；它们不是下一个字符的概率。')
    if not causal:
        print('当前关闭 Mask，仅用于对照；后面的真实 GPT 始终启用因果 Mask。')

token_options = [(f'{i}：{ch}',i) for i,ch in enumerate(PREFIX)]
attention_ui = panel('attention', render_attention, OrderedDict(
    query=widgets.Dropdown(options=token_options, value=3, description='谁在查询'),
    key=widgets.Dropdown(options=token_options, value=1, description='改谁的分数'),
    delta=widgets.FloatSlider(value=0, min=-3, max=3, step=.25, description='分数增量', continuous_update=False),
    causal=widgets.Checkbox(value=True, description='开启因果 Mask')))


<details><summary>核对与下一步</summary>未来位置先被 Mask 掉，分数怎么调都没有贡献。关闭 Mask 后，这条来源才能参与归一化。提高一个可见位置的分数，会改变整行权重，而非只改一个权重。现在我们得到了每个位置汇集来的内容；下一步解释它怎样写回原表示，以及为什么还需要 FFN。</details>

<details><summary>选读：这里的简化有什么边界？</summary>
手工投影没有训练，不应从它的热图推断猫和垫子的真实语义关系。权重反映这一层这一头的混合比例，也不等于完整模型预测的因果解释。Softmax 是标准方案，不代表所有注意力变体都必须采用它。点积缩放的直觉基于各分量尺度近似稳定等条件，不是对任意训练状态的严格保证。
</details>


## 4 · 组成 Block：交换信息、加工特征，并把更新写回

**位置**：从一头 Attention 走向完整 Transformer Block。**问题**：刚取回的是一份更新，还要表达多种匹配方式，并让模型逐步加工信息。

### 4.1 多头：同一份输入，使用多组投影

一头用一组 Q/K/V 投影；多头让多组投影分别算注意力，再拼接并投影回原来的向量维度。每头读完整输入的投影，不是简单把原输入切成几块，也没有预先指定“第一头学语法”。不同偏好可能从训练中产生。

后面的真实模型使用 `D=64`、`H=4`，每头 `d_k=16`。手算中一头只保留两维；真实模型中每头取回 16 维内容，四头拼成 64 维，再经输出投影得到 64 维更新，因此能加回原表示。下面的代码不调用 `nn.Transformer`，直接实现刚才的公式。


In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_head):
        super().__init__()
        assert d_model % n_head == 0
        self.n_head = n_head
        self.head_dim = d_model // n_head
        self.qkv = nn.Linear(d_model, 3*d_model)
        self.proj = nn.Linear(d_model, d_model)

    def forward(self, x, capture=False):
        B, T, D = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        # 按头整理已经投影好的坐标：(B,T,D) → (B,H,T,d_k)。
        q, k, v = [a.reshape(B,T,self.n_head,self.head_dim).transpose(1,2) for a in (q,k,v)]
        scores = (q @ k.transpose(-2,-1)) / math.sqrt(self.head_dim)
        allowed = torch.ones(T,T,device=x.device,dtype=torch.bool).tril()
        masked_scores = scores.masked_fill(~allowed, -torch.inf)
        weights = F.softmax(masked_scores, dim=-1)
        mixed = weights @ v
        joined = mixed.transpose(1,2).contiguous().reshape(B,T,D)
        update = self.proj(joined)
        trace = dict(q=q,k=k,v=v,scores=scores,masked_scores=masked_scores,
                     weights=weights,mixed=mixed,joined=joined,update=update) if capture else None
        return update, trace


### 4.2 FFN：读完上下文，再在每个位置加工特征

Attention 负责位置之间的信息交换。FFN（前馈网络，也称 MLP）则独立处理每个位置的向量：所有位置用同一套参数，但各自的输入已包含不同上下文。

$$\operatorname{FFN}(h)=W_2\,\operatorname{GELU}(W_1h+b_1)+b_2$$

这是一种按列向量书写的数学记法；PyTorch 用最后一维保存特征，完成相同的变换。一个神经元就是先算加权和再过激活函数，许多神经元组成一层。这里先从 64 维扩到 256 维，经非线性，再回到 64 维。非线性让它不只是两次矩阵乘法合成的一次线性变换。

FFN 不直接查询别的位置，但能加工 Attention 刚取来的组合特征。它和 Attention 都学习参数，不能把 Transformer 仅仅理解成“算一张注意力表”。

### 4.3 残差与 LayerNorm：保留原表示，并控制送入分支的尺度

**残差**把原向量与分支更新相加：`新表示 = 原表示 + 更新`。分支学习的是怎样补充或修改现有表示；这条直接通路也有助于深层训练，不保证每一维的信息都原样保留。

**LayerNorm**在每个位置的特征维度上归一化，再用可学习的缩放和偏移调整结果。它不跨位置平均，也不把句子变成概率。

本教程采用 **Pre-LN Block**：

$$u=x+\operatorname{MHA}(\operatorname{LN}_1(x)),\quad
x'=u+\operatorname{FFN}(\operatorname{LN}_2(u))$$

注意：残差通路中的 `x` 没有先被替换成归一化后的数。下一层接收 `x'`，再用新的参数重复交流与加工；输入字数不变，表示不断更新。前面的手算暂时直接用 X，真实 Block 会先做这里的 LayerNorm。


In [ ]:
class DecoderBlock(nn.Module):
    def __init__(self, d_model, n_head):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attention = CausalSelfAttention(d_model, n_head)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(nn.Linear(d_model,4*d_model), nn.GELU(), nn.Linear(4*d_model,d_model))

    def forward(self, x, capture=False):
        normalized = self.ln1(x)
        att_update, att_trace = self.attention(normalized, capture=capture)
        after_attention = x + att_update
        ffn_update = self.ffn(self.ln2(after_attention))
        output = after_attention + ffn_update
        trace = dict(input=x,normalized=normalized,attention=att_trace,
                     after_attention=after_attention,ffn_update=ffn_update,output=output) if capture else None
        return output, trace

# 用同一个四维教具检查形状与两次残差，参数仍未训练。
with torch.random.fork_rng():
    torch.manual_seed(19)
    demo_block = DecoderBlock(d_model=4,n_head=2)
with torch.no_grad():
    block_input = torch.tensor(toy_X,dtype=torch.float32)[None,:,:]
    block_out, block_trace = demo_block(block_input,capture=True)
vector_grid(block_trace['attention']['update'][0],list(PREFIX),'Attention 分支给出的更新')
vector_grid(block_trace['after_attention'][0],list(PREFIX),'第一次残差：原输入 + Attention 更新')
vector_grid(block_out[0],list(PREFIX),'第二次残差后：得到完整 Block 输出')
assert torch.allclose(block_trace['after_attention'],block_input+block_trace['attention']['update'])
assert torch.allclose(block_out,block_trace['after_attention']+block_trace['ffn_update'])

# 同一个最后位置的 FFN 算例：4 → 16 → 16 → 4；所有位置共用这些参数。
with torch.no_grad():
    ffn_input = demo_block.ln2(block_trace['after_attention'])[0,-1]
    ffn_expanded = demo_block.ffn[0](ffn_input)
    ffn_activated = demo_block.ffn[1](ffn_expanded)
    ffn_result = demo_block.ffn[2](ffn_activated)
table(['FFN 阶段','完整维度','前 4 项数值'],[
    [label,values.numel(),values[:4].round(decimals=3).tolist()]
    for label,values in [('归一化后的输入',ffn_input),('第一次线性变换',ffn_expanded),
                         ('GELU 非线性',ffn_activated),('第二次线性变换',ffn_result)]])
assert torch.allclose(ffn_result,block_trace['ffn_update'][0,-1],atol=1e-6)


In [ ]:
ln_example = torch.tensor([[1.,2.,3.,4.]])
centered = ln_example-ln_example.mean(dim=-1,keepdim=True)
manual_ln = centered/torch.sqrt(centered.square().mean(dim=-1,keepdim=True)+1e-5)
# 默认初始缩放 gamma=1，偏移 beta=0；训练会改变它们。
actual_ln = nn.LayerNorm(4)(ln_example)
table(['一行输入', '去均值并调整尺度', 'PyTorch 初始 LayerNorm'],
      [[ln_example.tolist()[0],manual_ln.tolist()[0],actual_ln.detach().tolist()[0]]])
assert torch.allclose(manual_ln,actual_ln,atol=1e-6)


**先预测**：假设 Attention 和 FFN 两个分支都输出全 0，一个 Pre-LN Block 输出什么？如果没有 Attention，只对带位置的向量反复做逐位置 FFN，最后的「在」能知道前面是猫还是狗吗？

<details><summary>核对与下一步</summary>两次残差都只加 0，Block 输出等于输入。单靠逐位置 FFN 无法把前文传给最后的位置；两句里「在」的初始字符和位置向量相同。现在有了完整 Block，下一步要把处理后的向量接到词表，才能真正预测文字。</details>


## 5 · 从隐藏向量到文字：补上真正的生成闭环

**位置**：走出多层 Block，来到输出头。**问题**：每个位置仍只有一排 64 个数，怎样对应到词表里的每一个字符？

最后再做一次 LayerNorm，然后用线性层把 64 维映射到 `V` 维。`V` 是词表大小，输出的每一项才对应一个候选字符的 **logit（原始分数）**。分数不是概率，可以为负。

对这些词表分数做 Softmax，得到下一个字符的分布。与第 3 节相比，公式相同，选项不同：

| Softmax 用在哪里 | 每个选项是什么 | 接下来做什么 |
|---|---|---|
| Attention 内部 | 当前输入中的一个位置 | 按权重混合该位置的 V 向量 |
| 语言模型输出 | 整个词表中的一个字符 | 选择一个字符，接回输入 |

模型为每个位置都计算一个词表分布。**位置 t 的分布预测 t+1**；生成新字符时只取最后一个位置，前面的分布留给训练使用。


In [ ]:
class MiniGPT(nn.Module):
    def __init__(self, vocab_size, d_model=64, n_head=4, n_layer=2, block_size=32):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size,d_model)
        self.pos_emb = nn.Embedding(block_size,d_model)
        self.blocks = nn.ModuleList([DecoderBlock(d_model,n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model,vocab_size,bias=False)

    def forward(self, ids, capture=False):
        if ids.ndim != 2 or not 1 <= ids.shape[1] <= self.block_size:
            raise ValueError(f'输入需要是 (B,T)，且 1 ≤ T ≤ {self.block_size}。')
        token_vectors = self.tok_emb(ids)
        position_vectors = self.pos_emb(torch.arange(ids.shape[1],device=ids.device))[None,:,:]
        x = token_vectors + position_vectors
        trace = dict(token_vectors=token_vectors,position_vectors=position_vectors,input=x,blocks=[]) if capture else None
        for block in self.blocks:
            x, block_trace = block(x,capture=capture)
            if capture:
                trace['blocks'].append(block_trace)
        normalized = self.ln_f(x)
        logits = self.head(normalized)
        if capture:
            trace.update(normalized=normalized,logits=logits)
            return logits, trace
        return logits

    @torch.no_grad()
    def inspect(self, ids):
        # 和训练/生成共用同一条 forward；并非另外编造一组可视化结果。
        return self(ids,capture=True)[1]

torch.manual_seed(7)
model = MiniGPT(VOCAB_SIZE).cpu()
training_status = '尚未训练'
with torch.no_grad():
    initial_logits = model(prefix_ids)
    initial_probs = F.softmax(initial_logits[0,-1],dim=-1)
initial_state = copy.deepcopy(model.state_dict())
roadmap({1,2,3})
print('输入 ID 形状：',tuple(prefix_ids.shape),'=（B：句子数，T：位置数）')
print('logits 形状：',tuple(initial_logits.shape),'=（B，T，V：词表大小）')
bars(initial_probs,CHARS,'未训练模型：最后位置的下一字符概率',top=8)


### 5.1 在真实模型里追踪同一位置

接下来全是这个 MiniGPT **实际算出的值**，不再使用四维手工参数。选择层、头和位置，顺着输入 → 匹配 → 混合 → 更新 → 词表概率看过去。为了便于阅读，向量表只显示前几列，计算使用完整维度。

“猫前缀”与“狗前缀”最后都为「在」，位置也相同；切换前缀时，字符与位置表给它的初始向量相同，经过 Attention 后的表示则可能不同。未训练时这些差异不代表已经学到了有用关系。第 7 节训练完成后，这个面板会刷新为训练后的模型，并显示状态。

**先预测**：选择位置 1「猫」，它的词表概率在预测「猫」本身，还是它后面的字符？它能读取位置 2「坐」吗？


In [ ]:
def inspect_real(prefix, layer, head, position):
    ids = encode(prefix)
    trace = model.inspect(ids)
    block = trace['blocks'][layer]
    att = block['attention']
    display(HTML(f'<b>真实 MiniGPT：{html.escape(training_status)}</b>；第 {layer+1} 层，第 {head+1} 头，位置 {position}「{html.escape(prefix[position])}」'))
    vector_grid(trace['token_vectors'][0,position],[prefix[position]],'查字符表（同 ID 总是同一行）')
    vector_grid(trace['position_vectors'][0,position],[f'位置 {position}'],'查位置表')
    vector_grid(block['input'][0],list(prefix),'这一层输入：第一层为两表之和，后续层为前一层输出')
    vector_grid(att['q'][0,head,position],[prefix[position]],'当前头的 Q；输入先经过 LayerNorm，再做投影')
    table(['来源位置','缩放后的匹配分数','Mask 后','注意力权重'],[
        [f'{j}：{ch}',float(att['scores'][0,head,position,j]),
         '不可读' if j>position else float(att['masked_scores'][0,head,position,j]),
         float(att['weights'][0,head,position,j])] for j,ch in enumerate(prefix)])
    vector_grid(att['k'][0,head],list(prefix),'当前头的 K：与 Q 匹配')
    vector_grid(att['v'][0,head],list(prefix),'当前头的 V：被权重混合的内容')
    vector_grid(att['mixed'][0,head,position],[prefix[position]],'当前头取回的内容：完整 16 维，只展示前 8 维')
    vector_grid(torch.stack([block['input'][0,position],att['update'][0,position],
                            block['after_attention'][0,position],block['ffn_update'][0,position],
                            block['output'][0,position]]),
                ['原输入','多头拼接并投影后的更新','第一次残差后','FFN 更新','完整 Block 输出'],
                '看同一位置怎样逐步更新')
    probs = F.softmax(trace['logits'][0,position],dim=-1)
    bars(probs,CHARS,f'经过全部层后的词表概率：位置 {position} 的输出预测下一位置',top=8)
    print('上面的词表概率始终来自完整模型；换“层/头”只改变中间观察窗口。')

real_ui = panel('real_model',inspect_real,OrderedDict(
    prefix=widgets.Dropdown(options=['小猫坐在','小狗坐在','小鸟飞在'],value=PREFIX,description='前缀'),
    layer=widgets.Dropdown(options=[('第 1 层',0),('第 2 层',1)],value=0,description='观察层'),
    head=widgets.Dropdown(options=[(f'第 {h+1} 头',h) for h in range(4)],value=0,description='观察头'),
    position=widgets.Dropdown(options=[(f'位置 {i}',i) for i in range(4)],value=3,description='观察位置')))


### 5.2 从分布选一个字符，再把它接回输入

**采样**按概率随机抽取候选；**贪心选择**直接取概率最大的一项。两者都不更新模型参数。

生成时可把词表 logits 除以温度 `T` 后再做 Softmax。较低温度让分布更集中，但只要仍采用采样，就不是“保证取最大值”。较高温度让分布更平。这里调的是**输出词表分布**的温度，不修改 Block 内的注意力公式。

下面每一步都列出本轮真正使用的前缀、概率和选中的字符。超过 32 个字符时，本教具只取最近 32 个，并将该窗口的位置从 0 重新编号；这是一种简单滑窗实现，不代表所有 GPT 的长上下文处理方式。


In [ ]:
@torch.no_grad()
def generate_steps(net, prompt, count=3, temperature=1., greedy=False, seed=11):
    if not math.isfinite(temperature) or temperature <= 0:
        raise ValueError('温度必须是正的有限数。')
    if not 1 <= count <= 12:
        raise ValueError('本实验每次生成 1 到 12 个字符。')
    ids = encode(prompt)
    rng = torch.Generator().manual_seed(seed)
    records = []
    for step in range(count):
        context = ids[:,-net.block_size:]
        logits = net(context)[0,-1]             # 只取最后一个位置。
        probs = F.softmax(logits/temperature,dim=-1)
        next_id = int(probs.argmax()) if greedy else int(torch.multinomial(probs,1,generator=rng).item())
        best = torch.topk(probs,min(3,VOCAB_SIZE))
        records.append(dict(context=decode(context[0]),token=ITOS[next_id],
                            probability=float(probs[next_id]),
                            top3='；'.join(f'{ITOS[int(i)]} {float(p):.3f}' for p,i in zip(best.values,best.indices))))
        ids = torch.cat([ids,torch.tensor([[next_id]])],dim=1)
    return decode(ids[0]),records

before_text,before_records = generate_steps(model,PREFIX)
table(['轮次','本轮输入窗口','前三个候选','本次选择','其概率'],
      [[i+1,r['context'],r['top3'],r['token'],r['probability']] for i,r in enumerate(before_records)],
      '未训练模型的真实生成轨迹')
print('接回后的文本：',before_text)


In [ ]:
def render_generation(prompt, temperature, greedy, count):
    text,records = generate_steps(model,prompt,count=count,temperature=temperature,greedy=greedy)
    display(HTML(f'<b>真实模型：{html.escape(training_status)}</b>；采样使用固定随机种子，方便比较温度。'))
    table(['轮次','输入窗口','前三候选','选择','概率'],[
        [i+1,r['context'],r['top3'],r['token'],r['probability']] for i,r in enumerate(records)])
    print('生成结果：',text)

generation_ui = panel('generation',render_generation,OrderedDict(
    prompt=widgets.Text(value=PREFIX,description='前缀'),
    temperature=widgets.FloatSlider(value=1.,min=.2,max=2.,step=.1,description='输出温度',continuous_update=False),
    greedy=widgets.Checkbox(value=False,description='贪心选择（不随机抽样）'),
    count=widgets.IntSlider(value=3,min=1,max=12,description='生成字数',continuous_update=False)))


<details><summary>核对与下一步</summary>位置 1「猫」的输出预测位置 2 的字符，但不能读取位置 2 的输入。在本例里正确标签为「坐」，模型当下的概率未必把它排在第一。最后位置的输出没有已知后续字符可读，只能根据前文预测。现在能续字了，下一步要问：这些参数怎样从随机数变得有用？</details>


## 6 · 学习：原文自己提供下一字的答案

**位置**：从“用固定参数预测”转到“用答案调整参数”。**问题**：我们怎样告诉模型哪些预测更好？

取已知文本「小猫坐在垫子上。」。输入去掉最后一个字，标签去掉第一个字，就得到一对错开一位的序列：

- 输入：`小 猫 坐 在 垫 子 上`
- 标签：`猫 坐 在 垫 子 上 。`

位置 0 用「小」预测「猫」；位置 3 用「小猫坐在」预测「垫」。一次前向能为所有位置产生分数，因此**训练可并行计算多个下一字任务**。因果 Mask 保证位置 t 不能读取 t+1 的答案；它不是把正确标签从词表里删掉。

生成时未来文本还没产生，所以要逐字符循环。训练时已知后续文本，可以把它放在张量里，但只能让各位置读取自己的前缀。


In [ ]:
lesson_ids = encode(SENTENCES[0])
lesson_x,lesson_y = lesson_ids[:,:-1],lesson_ids[:,1:]
table(['位置 t','本位置输入','允许读取的前缀','该输出的正确下一字'],[
    [t,ITOS[int(lesson_x[0,t])],decode(lesson_x[0,:t+1]),ITOS[int(lesson_y[0,t])]]
    for t in range(lesson_x.shape[1])])
assert torch.equal(lesson_x[:,1:],lesson_y[:,:-1])


### 6.1 怎样给一轮预测打分：交叉熵

如果正确字符是「垫」，模型给它的概率为 `p`，这一位置的损失就是 `−log(p)`。给正确答案的概率越高，损失越低：`p=0.5` 时约 0.693，`p=0.1` 时约 2.303。损失不只检查最终抽到了什么字符，而是检查整个预测分布给正确答案留了多少概率。

把所有训练位置的损失取平均，得到这一批次的交叉熵。PyTorch 的 `cross_entropy` **直接接收 logits**，内部用稳定的 log-softmax 计算；不要先把概率再传进去。


In [ ]:
with torch.no_grad():
    lesson_logits = model(lesson_x)
    log_probs = F.log_softmax(lesson_logits,dim=-1)
    target_log_probs = log_probs.gather(-1,lesson_y.unsqueeze(-1)).squeeze(-1)
    per_position_loss = -target_log_probs
    manual_loss = per_position_loss.mean()
    builtin_loss = F.cross_entropy(lesson_logits.reshape(-1,VOCAB_SIZE),lesson_y.reshape(-1))
table(['位置','正确下一字','模型分配给它的概率','该位置 −log(p)'],[
    [t,ITOS[int(lesson_y[0,t])],float(target_log_probs[0,t].exp()),float(per_position_loss[0,t])]
    for t in range(lesson_x.shape[1])])
print('手算平均损失：',float(manual_loss),'；cross_entropy：',float(builtin_loss))
assert torch.allclose(manual_loss,builtin_loss,atol=1e-6)


### 6.2 错误怎样改动参数：一次前向、反向、更新

梯度给出损失对每个参数的局部变化率。最基本的梯度下降用 `参数 ← 参数 − 学习率 × 梯度`，往局部降低损失的方向走。学习率决定每次改多少；它与生成温度是两回事。

反向传播把输出损失沿着同一条计算链传回输出头、FFN、Attention 和两张输入表。我们不会手工规定「猫应该看谁」，而是让预测损失调整产生这些匹配和内容的参数。本教程用 AdamW，它会结合梯度历史调整更新步长，还带权重衰减，因此不必等于最简单的单步梯度下降公式。

下面复制一份模型，只做一次更新，观察真实损失和参数变化。副本用于这个实验，下一节完整训练从原模型的初始参数开始。一次更新不保证所有位置都变好，也不保证从此总能生成目标句子。


In [ ]:
step_model = copy.deepcopy(model)
step_optimizer = torch.optim.AdamW(step_model.parameters(),lr=1e-3)
tracked_names = ['tok_emb.weight','pos_emb.weight','blocks.0.attention.qkv.weight','blocks.0.ffn.0.weight','head.weight']
tracked_before = {name:p.detach().clone() for name,p in step_model.named_parameters() if name in tracked_names}

step_logits = step_model(lesson_x)                       # 前向：每个位置预测下一字。
step_loss = F.cross_entropy(step_logits.reshape(-1,VOCAB_SIZE),lesson_y.reshape(-1))
step_optimizer.zero_grad(set_to_none=True)               # 清掉上轮梯度。
step_loss.backward()                                    # 反向：计算每个参数的梯度。
grad_norms = {name:float(p.grad.norm()) for name,p in step_model.named_parameters() if name in tracked_names}
step_optimizer.step()                                   # 真正改动参数。

with torch.no_grad():
    loss_after_one_step = F.cross_entropy(step_model(lesson_x).reshape(-1,VOCAB_SIZE),lesson_y.reshape(-1))
table(['参数','这次梯度的范数','更新前后差值的范数'],[
    [name,grad_norms[name],float((p-tracked_before[name]).norm().detach())]
    for name,p in step_model.named_parameters() if name in tracked_names])
print(f'这次运行：更新前 loss={float(step_loss.detach()):.4f}，更新后 loss={float(loss_after_one_step):.4f}')


**先预测**：把 `optimizer.step()` 去掉，loss 仍能算出来吗？参数会变吗？如果把右移标签改成输入本身，模型学到的任务还是“预测下一字”吗？

<details><summary>核对与下一步</summary>仍能前向算 loss、反向算梯度，但不执行更新就不会改参数。标签等于输入会变成在当前位置复述输入，与下一字任务不同。现在我们看清一次学习的全过程；下一节只是把正确的这个过程重复多次。</details>


## 7 · 组装与训练：把同一条链重复运行

**位置**：从单条样本走向多个训练窗口。**问题**：怎样得到一批包含不同上下文的下一字任务，并观察学习前后的实际变化？

我们已经逐节定义了 Attention、Block、输出头及完整模型；这里直接使用同一个 MiniGPT，没有另外藏一套训练专用实现。训练批次从练习语料中取长度 32 的窗口，标签整体右移一位。`B=16` 表示一批有 16 个窗口，`T=32` 表示每个窗口 32 个位置，`D=64` 是每个位置的表示宽度。

全模型有 2 个 Block、每层 4 个头，使用可学习位置表；没有加入 Dropout，以便观察和核对数值。我们训练 180 步，显示批次损失，再用同一组已知练习句子比较训练前后。**这组句子也在训练语料中，比较只说明训练集拟合，不是验证集成绩。**

本单元每次运行都会重置为前面保存的初始参数，并重置抽样随机数，方便重复实验。控件面板随后刷新为训练后的实际模型。运行时间取决于 CPU。


In [ ]:
model.load_state_dict(initial_state)
model.train()
training_status = '训练中'
training_data = encode(CORPUS)[0]
batch_rng = torch.Generator().manual_seed(23)

def get_batch(batch_size=16,block_size=32):
    # start+block_size 仍须有一个字符，供最后位置作为标签。
    starts = torch.randint(len(training_data)-block_size,(batch_size,),generator=batch_rng)
    x = torch.stack([training_data[i:i+block_size] for i in starts])
    y = torch.stack([training_data[i+1:i+block_size+1] for i in starts])
    return x,y

@torch.no_grad()
def practice_loss(net):
    total, count = 0.,0
    for text in SENTENCES:
        ids = encode(text)
        x,y = ids[:,:-1],ids[:,1:]
        total += float(F.cross_entropy(net(x).reshape(-1,VOCAB_SIZE),y.reshape(-1),reduction='sum'))
        count += y.numel()
    return total/count

before_practice_loss = practice_loss(model)
before_text,before_records = generate_steps(model,PREFIX)  # 重跑此单元时，训练前续写也从初始参数重新计算。
with torch.no_grad():
    before_training_probs = F.softmax(model(prefix_ids)[0,-1],dim=-1).clone()
optimizer = torch.optim.AdamW(model.parameters(),lr=3e-3)
loss_history = []
progress = display(HTML('开始训练……'),display_id=True)
for step in range(1,181):
    xb,yb = get_batch()
    logits = model(xb)
    loss = F.cross_entropy(logits.reshape(-1,VOCAB_SIZE),yb.reshape(-1))
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    loss_history.append(float(loss.detach()))
    if step == 1 or step % 30 == 0:
        progress.update(HTML(f'<b>step {step}/180；当前批次 loss={loss_history[-1]:.4f}</b>'+loss_plot(loss_history)))

model.eval()
training_status = '已完成 180 步训练'
after_practice_loss = practice_loss(model)
with torch.no_grad():
    after_training_probs = F.softmax(model(prefix_ids)[0,-1],dim=-1).clone()
after_text,after_records = generate_steps(model,PREFIX)
table(['对照项','训练前','训练后'],[
    ['已知练习句子的逐字符平均交叉熵',before_practice_loss,after_practice_loss],
    ['同前缀、同随机种子的 3 字续写',before_text,after_text]])
for key in ['real_model','generation']:
    PANELS[key]['refresh']()


In [ ]:
# 同一个前缀、同一完整词表：对比的是实际模型分布。
bars(before_training_probs,CHARS,'训练前：小猫坐在 → 下一字符',top=8)
bars(after_training_probs,CHARS,'训练后：小猫坐在 → 下一字符',top=8)
table(['轮次','本轮输入','前三候选','选择','概率'],[
    [i+1,r['context'],r['top3'],r['token'],r['probability']] for i,r in enumerate(after_records)],'训练后生成闭环')
print('再回第 5 节，切换猫/狗前缀、层和头；面板当前使用训练后的参数。')


### 7.1 核对模型确实遵守我们讲的规则

“跑完没有报错”还不够。下面核对完整前向形状、每行权重的和、未来位置的零权重、V 的混合、两次残差、输出投影，以及改动未来字符对更早输出的影响。它们检查计算关系，不要求某个头一定看「猫」，也不要求随机采样固定续出「垫子上」。


In [ ]:
trace = model.inspect(prefix_ids)
T = prefix_ids.shape[1]
assert trace['logits'].shape == (1,T,VOCAB_SIZE)
assert torch.allclose(trace['input'],trace['token_vectors']+trace['position_vectors'])
for block in trace['blocks']:
    att = block['attention']
    assert torch.allclose(att['weights'].sum(-1),torch.ones(1,4,T),atol=1e-6)
    future = torch.ones(T,T,dtype=torch.bool).triu(1)
    assert torch.count_nonzero(att['weights'][...,future]) == 0
    assert torch.allclose(att['mixed'],att['weights']@att['v'],atol=1e-6)
    assert torch.allclose(block['after_attention'],block['input']+att['update'],atol=1e-6)
    assert torch.allclose(block['output'],block['after_attention']+block['ffn_update'],atol=1e-6)
assert torch.allclose(trace['logits'],model.head(trace['normalized']),atol=1e-6)

# 修改位置 2、3 的输入，位置 0、1 的输出应保持不变。
changed = prefix_ids.clone()
changed[0,2:] = torch.tensor([STOI['狗'],STOI['门']])
with torch.no_grad():
    assert torch.allclose(model(prefix_ids)[:,:2],model(changed)[:,:2],atol=1e-6)
xb_check,yb_check = get_batch(batch_size=2)
assert torch.equal(xb_check[:,1:],yb_check[:,:-1])
assert all(math.isfinite(x) for x in loss_history)
assert math.isfinite(after_practice_loss)
print('核对通过：形状、归一化、因果隔离、内容混合、残差、输出映射和标签右移。')
print(f'已知练习句子的实际 loss：{before_practice_loss:.4f} → {after_practice_loss:.4f}')


**先预测并解释**：如果训练 loss 降得很低，能不能说明它已经学会自然语言？如果某一注意力头的热图不符合你的直觉，能不能直接判断这个模型训练失败？

<details><summary>核对</summary>这几句话高度重复，低训练 loss 可能主要来自记忆。需要独立数据才能评价泛化。模型还有其他头、其他层、FFN 和残差；单张热图不足以判断整体是否学好。我们用它追踪计算，而不是给每个头强行贴上固定职责。</details>


## 8 · 带着已经走通的数据流，再看 Transformer 家族

**位置**：回到全景。**问题**：同样的“表示 → 交流 → 加工”，怎样用于理解、填空和翻译？

| 结构 | 位置能读取谁 | 输入输出怎样连接 | 常见任务 |
|---|---|---|---|
| Encoder（如 BERT） | 通常可读取输入两侧 | 输入变为上下文表示，接分类等任务头 | 分类、实体识别、遮盖预测 |
| Decoder-only（本教程、GPT） | 只能读取自己与过去 | 前缀变为下一 token 分布，循环生成 | 续写、语言建模 |
| Encoder-Decoder（原版 Transformer） | Encoder 读源文本；Decoder 自注意力读已生成前缀 | Decoder 还用交叉注意力读取 Encoder 的输出 | 翻译等序列到序列任务 |

交叉注意力里，Q 来自 Decoder 当前表示，K/V 来自 Encoder 输出；它解决“生成目标文本时怎样读取源文本”的问题。GPT 没有因此缺少一个尚未讲的 Encoder；它的主线就是我们已经实现的因果 Block 堆叠。

### 8.1 真实模型与本教具的几个差异

- **分词**：真实模型常把词、子词或字节片段作为 token，不一定一字一个。词表改变了输入和输出头的大小，ID → 向量 → Block → 词表分数的链条仍然成立。
- **位置**：本教程使用可学习位置表。原版 Transformer 使用正弦/余弦位置编码；其他模型也有 RoPE 等做法，它们怎样影响计算需要分别解释，不能直接等同于本教程的位置相加。
- **规模与数据**：更大的模型需要更多数据、训练策略和评估；扩规模不会自动把小语料训练成绩变成泛化证据。
- **工程**：高效生成可缓存已经算过的 K/V。本教程为看清循环每次重算整个窗口；没有实现缓存、工具调用或对话微调。

### 8.2 试着不用术语讲完这条链

给自己五个问题，每个都用本文件里的一个数值或实验回答：

1. 为什么 ID 不是文字的意义？字符向量与上下文化后的向量有什么不同？
2. Attention 的权重乘的是什么，输出是什么？
3. Attention、FFN、残差和 LayerNorm 各承担什么工作？
4. 为什么向量还需要输出头？注意力权重与下一字概率的选项分别是什么？
5. 训练标签从哪来，哪些参数会变？生成时又有哪些量会变？

遇到卡点时，回到相应的输入、输出和实验。目标是能沿一个具体例子解释全过程，而不只是认出公式的名字。

### 参考与实现核对

- Vaswani 等：[Attention Is All You Need](https://arxiv.org/abs/1706.03762)。原论文是 Encoder-Decoder；本教程用 Decoder-only 主线解释其核心组件，采用 Pre-LN 与可学习位置表，并非逐项复刻原论文。
- PyTorch：[cross_entropy](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.cross_entropy.html)，核对 logits 输入与目标 ID。
- PyTorch：[LayerNorm](https://docs.pytorch.org/docs/stable/generated/torch.nn.LayerNorm.html)，核对归一化维度、方差和可学习缩放/偏移。

**本文件的边界**：所有训练、算例和交互均可离线运行；只有上述参考链接需要网络。关闭内核后再打开文件，需要重新运行相应代码才能操作控件。本教程不写模型检查点或下载缓存。
